# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the [mlcroissant](https://github.com/mlcommons/croissant) library.

The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with assessment scores (e.g., GAD-7, PHQ-9, PSQ).

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the dataset metadata and records from the specified Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a Python dict
metadata = dataset.metadata.to_json()
print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview

Let's inspect the record sets (tables) in the dataset, their `@id`s, and the available fields for each record set. All `@id` references are derived directly from the metadata.

In [ ]:
from pprint import pprint

# Get all available record sets and their IDs
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    rs_meta = rs.to_json()
    print(f"- Name: {rs_meta.get('name', '')}")
    print(f"  @id: {rs_meta.get('@id')}")
    print(f"  Description: {rs_meta.get('description','').strip()}")
    fields = rs_meta.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Field @ids:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id')}")
        else:
            print(f"    - {f}")
    print()
    # For a dense overview, print first 1 record per record set with field @id mapping
    try:
        record = next(dataset.records(record_set=rs_meta['@id']))
        print(f"  Sample record keys: {list(record.keys())}")
        print(f"  Sample record: {json.dumps(record, indent=2)[:400]} ...\n")
    except StopIteration:
        print("  (No records found)\n")

## 3. Data Extraction

We will extract data from the main record set. We'll use its `@id` and each field by its `@id` for clarity and reproducibility.

Below, we demonstrate extraction from all record sets. The main survey data is typically found under the most prominent record set (often named 'main', 'survey_responses', etc.). Use the first record set if unsure.

In [ ]:
# List all record set @ids
record_set_ids = [rs.to_json().get('@id') for rs in dataset.record_sets]
print(f'RecordSet @ids: {record_set_ids}')

# We'll load all record sets into DataFrames, referencing by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # If not empty
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}. Columns:")
        print(df.columns.to_list())
    else:
        print(f"No records found for RecordSet {record_set_id}.")

# Use the first record set as the main one for demonstration:
main_record_set_id = record_set_ids[0]
print(f"\nUsing main record set: {main_record_set_id}\n")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some basic data processing:
- Filter records based on a numeric field (using GAD-7 or PHQ-9 score if available)
- Normalize this field
- Optionally, group the data by an attribute such as gender or education level and inspect basics.

In [ ]:
# Choose a numeric field for demonstration (use the correct @id key as used in the DataFrame)
main_df = dataframes[main_record_set_id]

# Try to determine the GAD-7 or PHQ-9 columns (if unsure, print columns)
print("First 15 column names for inspection:")
print(main_df.columns[:15].to_list())

# Examples - replace with correct @id as per 'main_df.columns' list
candidate_numeric_fields = [col for col in main_df.columns if any(s in col.lower() for s in ['gad', 'phq', 'score', 'psq', 'total'])]
print(f"Candidate numeric fields: {candidate_numeric_fields}")

# Let's pick the first numeric field candidate
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else main_df.select_dtypes(include='number').columns[0]
print(f"Using numeric field: {numeric_field}")

threshold = main_df[numeric_field].mean()  # Use mean as threshold
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a common categorical field (e.g., gender, education)
candidate_group_fields = [col for col in main_df.columns if any(x in col.lower() for x in ['gender', 'sex', 'education', 'village', 'marital'])]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    print(f"Grouping filtered data by {group_field} and showing mean statistics:")
    grouped_df = (
        filtered_df.groupby(group_field)[[numeric_field, f"{numeric_field}_normalized"]].mean().sort_values(numeric_field, ascending=False)
    )
    display(grouped_df)
else:
    print("No suitable group field found to demonstrate grouping.")

## 5. Visualization

Let's plot the distribution of the selected numeric field and visualize group differences if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If grouped by a field, visualize group differences (boxplot)
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We demonstrated how to:
- Programmatically inspect and extract survey records using the `mlcroissant` library,
- Filter and process records using field `@id`s for transparency and reproducibility,
- Visualize field distributions and group statistics,
- Prepare the dataset for further data science and AI/ML tasks.

This notebook can be easily adapted for other Croissant datasets by changing only the schema URL. For advanced tasks, refer to the [`mlcroissant` API documentation](https://croissant-mlcommons.readthedocs.io/).
